# NASA C-MAPSS Turbofan Engine RUL Prediction

## 1. Problem Definition & Overview

### Objective
The goal of this project is to predict the **Remaining Useful Life (RUL)** of aircraft turbofan engines from multivariate sensor measurements. Reliable RUL estimates are central to predictive maintenance: they help prevent in-flight failures while avoiding unnecessary scheduled maintenance.

### Background & Context
Aircraft engines degrade gradually through use. Prognostics and Health Management (PHM) is the discipline of estimating how much longer a system can operate before failure.

We use the benchmark **NASA C-MAPSS (Commercial Modular Aero-Propulsion System Simulation)** dataset, specifically subset **FD001**.

In the FD001 sub-dataset:
- **Operating Conditions:** Single nominal cruise condition (Sea Level).
- **Fault Mode:** High-Pressure Compressor (HPC) degradation (flow and efficiency loss).
- **Data Generation:** Simulated using a physics-informed thermodynamic turbofan model with exponential damage propagation ($h(t) = 1 - d - \exp(at^b)$) and superimposed operational process noise.

## 2. Dataset Overview

C-MAPSS simulates run-to-failure trajectories for a fleet of turbofan engines:
- **Train Data (`train_FD001.txt`):** Full run-to-failure trajectories. Each engine starts healthy and degrades until failure.
- **Test Data (`test_FD001.txt`):** Trajectories that stop *before* failure.
- **RUL Data (`RUL_FD001.txt`):** The true remaining cycles for each test engine at its truncation point.

### Data Schema
Each record has 26 columns:
1. `engine_id`: unit number (1 to N)
2. `cycle`: flight cycle index within the engine's trajectory
3. `os_1`, `os_2`, `os_3`: three operating settings
4. `sensor_1` to `sensor_21`: 21 sensor measurements

## 3. Data Loading

The raw files are whitespace-separated text with no header, so we load them with `sep=r'\s+'` and `header=None`.

In [ ]:

import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
train_data = pd.read_csv("data/raw/train_FD001.txt", sep=r"\s+", header = None)
test_data = pd.read_csv("data/raw/test_FD001.txt", sep =r"\s+", header = None)
RUL_data = pd.read_csv("data/raw/RUL_FD001.txt", sep =r"\s+", header = None)

## 4. Data Audit

Before any analysis, we check both files for size, missing values, duplicates, and column types. Any problem found here would affect every later step.

### 4.1 Training Set

In [ ]:
train_data.shape

In [ ]:

train_data.head()

In [ ]:
train_data.info()

In [ ]:
train_data.isnull().sum()

In [ ]:
train_data.duplicated().sum()

**Findings:** 20,631 rows x 26 columns. No missing values, no duplicates. Types are a mix of `int64` and `float64`. The training set is clean and ready to use.

### 4.2 Test Set

In [ ]:
test_data.shape


In [ ]:
test_data.head()

In [ ]:
test_data.info()

In [ ]:
test_data.isnull().sum()

In [ ]:
test_data.duplicated().sum()

**Findings:** 13,096 rows x 26 columns, likewise free of missing values and duplicates.

## 5. Column Naming


The raw datasets do not contain column headers. We assign standard schema names for `engine_id`, `cycle`, operational settings (`os_1` to `os_3`), and sensors (`sensor_1` to `sensor_21`).

In [ ]:
# adding name to train and test data columns
column_names = (
    ["engine_id", "cycle"]
    + [f"os_{i}" for i in range(1, 4)]
    + [f"sensor_{i}" for i in range(1, 22)]
)
train_data.columns = column_names
test_data.columns = column_names

## 6. Exploratory Data Analysis (EDA)

### 6.1 Engine Lifetime

We examine the operational lifetime (maximum cycles) of engines in the training and test sets to understand degradation timeframes.

In [ ]:

train_data["engine_id"].nunique() # number of unique engines in the training dataset

In [ ]:
test_data["engine_id"].nunique() # number of unique engines in the test dataset

#### Distribution of Engine Lifetime

In [ ]:
engine_life = train_data.groupby("engine_id")["cycle"].max() #maximum number of cycles for each engine in the training dataset
sns.histplot(engine_life, bins=30, kde=True)
plt.title("Distribution of Engine Lifetime in Training Dataset")

In [ ]:
engine_life.describe()

**Train set:** the distribution is roughly bell-shaped, peaking around 180-220 cycles. Lifetimes span ~130 to ~360 cycles, with a mean of ~206.

In [ ]:
test_engine_life =test_data.groupby("engine_id")["cycle"].max() #maximum number of cycles for each engine in the test dataset
sns.histplot(test_engine_life, bins=30, kde=True)
plt.title("Distribution of the last cycle in Test Dataset")

In [ ]:
sns.boxplot(x=engine_life)


**Test set:** the last observed cycle is again roughly normally distributed, with most engines stopping between 150-200 cycles. As expected, these are shorter than full training lifetimes because test trajectories end before failure.

### 6.2 Sensor Audit & Distributions

Next we profile all 21 sensors: summary statistics, distributions, and correlations. This tells us which sensors actually vary and how they relate to each other.

In [ ]:
sensor_df = train_data.iloc[:, 5: ] #selecting only the sensor columns
pd.set_option('display.max_columns', None) #to display all columns in the dataframe
sensor_df.describe().T

#### Sensor Distributions

In [ ]:

fig, axes = plt.subplots(7, 3, figsize=(18, 30))
for i, sensor in enumerate(sensor_df.columns):
    sns.histplot(sensor_df[sensor], bins=30, kde=False, ax=axes[i//3, i%3])

In [ ]:
fig, axes = plt.subplots(7, 3, figsize=(18, 30))
for i, sensor in enumerate(sensor_df.columns):
    sns.boxplot(x=sensor_df[sensor], ax=axes[i//3, i%3])

In [ ]:
sensor_corr = sensor_df.corr()
fig , ax = plt.subplots(figsize=(12, 10))
sns.heatmap(sensor_corr, annot=True, fmt=".2f", cmap="coolwarm")

#### Invariant Sensors Identified (Zero Standard Deviation)
- `sensor_1` ($T_2$ - Total temperature at fan inlet)
- `sensor_5` ($P_2$ - Total pressure at fan inlet)
- `sensor_10` ($epr$ - Engine pressure ratio $P_{50}/P_2$)
- `sensor_16` ($P_{15}$ - Total pressure in bypass-duct)
- `sensor_18` ($NR_f$ - Demanded corrected fan speed)
- `sensor_19` ($NR_c$ - Demanded corrected core speed)

#### Physical & Engineering Rationale
In the **FD001** simulation setup, flight conditions (Altitude, Mach number, and Throttle Resolver Angle) are held at a single steady cruise operating condition (Sea Level). Consequently, inlet ambient conditions ($T_2$, $P_2$) and closed-loop control targets ($NR_f$, $NR_c$, $epr$) remain thermodynamically constant. Because they carry zero variance and no degradation signature of the HPC module, they are pruned from the feature space.

#### Remaining Sensors
The remaining 15 variable sensors capture progressive thermal and mechanical degradation (such as rising Exhaust Gas Temperature $T_{50}$ / `sensor_4`, dropping HPC stall margin, and increasing fuel-air ratio `sensor_12`).

**Key observations:**
- Sensors 1, 5, 10, 16, 18, and 19 show essentially zero variation and provide no useful information.
- The remaining 15 sensors have varied distribution shapes (normal, skewed, multi-modal).
- Sensors 9 and 15 are highly correlated with each other.
- The constant sensors will be removed during preprocessing.

### 6.3 Sensor Behavior Across Cycles

Distributions alone do not show degradation. Plotting each sensor against cycle for a few sample engines reveals whether sensors trend as the engine wears.

In [ ]:
fig, axes = plt.subplots(7, 3, figsize=(18, 30))

# the change of sensor's cycle in 3 engine
for engine_id in range(1, 4):
    engine_df = train_data[train_data["engine_id"] == engine_id]
    
    for j, sensor in enumerate(sensor_df.columns):
        sns.scatterplot(
            data=engine_df, x="cycle", y=sensor, ax=axes[j//3, j%3],
            label=f"Engine {engine_id}", alpha=0.6
        )

plt.tight_layout()

**Key observations:**
- Stable sensors (1, 5, 10, 16, 18, 19) stay flat across all cycles, confirming they carry no degradation information.
- Most other sensors show clear upward or downward trends over time, capturing engine wear.
- Engines share the general trend but differ in starting points and slopes, reflecting different degradation rates.

### 6.4 Ground-Truth Test RUL & Operating Settings

The RUL file holds one true label per test engine. We audit its structure and distribution before using it later as `y_test`. We also examine the three operating settings recorded at every snapshot.

In [ ]:
RUL_data.describe()

In [ ]:
sns.histplot(RUL_data, bins= 30, kde=True)

**Ground-truth test RUL:** 100 values, ranging from 7 to 145 cycles remaining.

A quick structural check of the RUL file:

In [ ]:
RUL_data.head()

In [ ]:
RUL_data.shape

In [ ]:
RUL_data.isnull().sum()

In [ ]:
RUL_data.info()

**Findings:** 100 rows (one per test engine), no missing values, integer type.

The column has no name yet, so we rename it to `RUL`:

In [ ]:
RUL_data.columns = ["RUL"]

#### Operating Settings
Finally, we summarize the three operating settings to see whether any of them are constant under the FD001 flight condition.

In [ ]:
os_data = train_data[['os_1', 'os_2', 'os_3']]
os_data.describe().T

In [ ]:
# Effect of operating settings on sensor behavior
fig , ax = plt.subplots(figsize=(12, 10))
sns.heatmap(train_data.corr(), annot=True, fmt=".2f", cmap="coolwarm")

#### Operating Settings Analysis (`os_1`, `os_2`, `os_3`)
- **Observation:** `os_3` exhibits virtually zero variance across all cycles in the FD001 dataset, reflecting a single fixed power-setting simulation.
- **Decision:** `os_3` is excluded from the model feature matrix. `os_1` and `os_2` exhibit slight nominal variations due to simulated ambient noise and are standardized accordingly.

## 7. Constructing the RUL Target

The training file contains no explicit labels, but every training trajectory runs to failure. That lets us compute the target for each observation directly:

$$\text{RUL} = \text{Max Cycle of Engine} - \text{Current Cycle}$$

An engine's final recorded cycle has RUL = 0; earlier rows get the number of cycles remaining after them.

Applied to the whole training set with a grouped transformation:

In [ ]:

max_cycles = train_data.groupby('engine_id')['cycle'].transform('max')

train_data['RUL'] = max_cycles - train_data['cycle']

In [ ]:
train_data.head()

In [ ]:
sns.histplot(train_data['RUL'])

In [ ]:
sns.scatterplot(x = train_data['cycle'], y = train_data['RUL'])
# all the engine follow the same pattern, as the cycle increases,
# the RUL  decrese.


## 8. Validation Strategy (Engine-Based Split)

In time-series sensor data from complex machinery, adjacent cycles from the same engine are highly correlated. A standard random row-level split would place cycles from the same engine into both training and validation sets, causing **data leakage** and overly optimistic performance estimates.

To prevent this, we split our data at the **`engine_id` level** (80% training engines, 20% validation engines).

In [ ]:
engine_id = train_data['engine_id'].unique()

In [ ]:
# split the engines id 
train_engines, val_engines = train_test_split(
    engine_id,
    test_size=0.2,
    random_state=42
)

# make train and validation dataframe

validation_df = train_data[train_data["engine_id"].isin(val_engines)]

train_df = train_data[train_data["engine_id"].isin(train_engines)]

In [ ]:
# check the number of engine
validation_df['engine_id'].nunique(),train_df['engine_id'].nunique()

## 9. Preprocessing & Feature Selection

Based on the audit and EDA, we drop everything that cannot help the model:
- **Constant sensors:** `sensor_1, 5, 10, 16, 18, 19` (zero variance).
- **Constant operating setting:** `os_3`.
- **Identifiers:** `engine_id` and `cycle`. The model must learn degradation from sensor physics, not from counting cycles or unit numbers.
- **Outliers:** kept (see below).

In [ ]:
# Remove Constant Sensors
removed_list = ['cycle','engine_id','os_3','sensor_1','sensor_5','sensor_10','sensor_16','sensor_18','sensor_19']

train_df = train_df.drop(removed_list, axis= 1)
validation_df = validation_df.drop(removed_list, axis= 1)

# we keep cycle, engine_id to make sure the order is right, it will be remove later
test_data = test_data.drop(removed_list[2:], axis= 1)



### Outlier Analysis
Boxplots and violin plots per feature show the spread of each retained sensor.

In [ ]:
fig, axes = plt.subplots(6, 3, figsize=(18, 30))
for i, item in enumerate(train_df.columns):
    sns.boxplot(x=train_df[item], ax=axes[i//3, i%3])

**Why outliers stay:** extreme readings here are not data errors. They reflect real operating variation and late-stage degradation toward end-of-life. Removing them would discard exactly the information RUL prediction depends on.

In [ ]:
fig, axes = plt.subplots(6, 3, figsize=(18, 30))
for i, item in enumerate(train_df.columns):
    sns.violinplot(x=train_df[item], ax=axes[i//3, i%3])

One borderline case remains: `sensor_6` takes very few distinct values and barely varies.

In [ ]:
train_df["sensor_6"].nunique()

In [ ]:
train_df["sensor_6"].value_counts()

`sensor_6` exhibits minimal variance and contributes very little signal for the model to learn from, so it is dropped from all three datasets.

In [ ]:
# remove sensor 6
train_df = train_df.drop(["sensor_6"], axis = 1)
validation_df = validation_df.drop(["sensor_6"], axis = 1)
test_data = test_data.drop(["sensor_6"], axis = 1)

### Reducing the Test Set
Every test engine stops before failure, and the ground truth in `RUL_FD001.txt` refers to the engine state at that final cycle. We therefore keep only the last recorded cycle per engine. `engine_id` and `cycle` stay in the frame a little longer to guarantee correct ordering and alignment; they are removed when the final feature matrix is built.

In [ ]:
test_data = test_data.sort_values(['engine_id', 'cycle']).groupby('engine_id').tail(1)

## 10. Standardization (Scaling)

We standardize the retained sensors and operating settings with `StandardScaler`.

The scaler is **fitted exclusively on the training set** (`train_df`), and then used to **transform** the validation and test sets.


In [ ]:
scaler = StandardScaler()

scale_cols = ['os_1', 'os_2', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_7', 
              'sensor_8', 'sensor_9', 'sensor_11', 'sensor_12', 'sensor_13',
              'sensor_14', 'sensor_15', 'sensor_17', 'sensor_20', 'sensor_21']

train_df[scale_cols] = scaler.fit_transform(train_df[scale_cols])
validation_df[scale_cols] = scaler.transform(validation_df[scale_cols])
test_data[scale_cols] = scaler.transform(test_data[scale_cols])

## 11. Final Dataset Summary & Ready for Modeling

#### Exported artifacts (saved to `data/processed/`)
- `X_train.csv` & `y_train.csv`: training feature matrix (16 features, 80 engines) with RUL targets.
- `X_val.csv` & `y_val.csv`: validation split (20 holdout engines) for unbiased tuning.
- `X_test.csv` & `y_test.csv`: one terminal observation per test engine (100 total), aligned with the true RUL file.

In [ ]:
X_train = train_df.drop(columns=["RUL"])
y_train = train_df["RUL"]

X_val = validation_df.drop(columns=["RUL"])
y_val = validation_df["RUL"]

In [ ]:
test_data = (
    test_data
    .sort_values(["engine_id", "cycle"])
    .groupby("engine_id")
    .tail(1)
    .sort_values("engine_id")
    .reset_index(drop=True)
)



X_test = test_data.drop(columns=["engine_id", "cycle"])
y_test = RUL_data

assert len(X_test) == len(y_test)

In [ ]:

df_list = {
    'X_train': X_train,
    'y_train': y_train,
    'X_val': X_val,
    'y_val': y_val,
    'X_test': X_test,
    'y_test': y_test
}

for name, df in df_list.items():
    df.to_csv(f'data/processed/{name}.csv', index=False)